# Tutorial: GB connection intelligence workflow

This notebook demos the two connection-intelligence MCP tools added in `luminus-py` 0.5.0. It runs a site connection report against a GB coordinate, reads the queue, headroom, and land-constraint traffic lights, inspects the canonical top TEC entries that feed the report, and then runs a Gate 2 readiness checklist against a hypothetical project.

Neither tool is a connection offer, a commercial recommendation, or a prediction of a Gate 2 decision. The site report is a composition over public data snapshots; the Gate 2 check is a rules-based review of publicly documented entry criteria. Always verify current requirements with the relevant network operator and NESO before acting on any output.


## Outline

1. Start a connections-profile Luminus client.
2. Run the site connection report for a GB coordinate.
3. Read the traffic lights and the markdown summary.
4. Inspect the canonical top TEC entries and DNO headroom entry.
5. Review the confidence notes for upstream-failure context.
6. Run a Gate 2 readiness check for the same hypothetical project.
7. Read the per-rule checklist and status counts.


In [ ]:
from __future__ import annotations

from luminus import (
    Gate2ReadinessCheckSnapshot,
    Luminus,
    SiteConnectionReportSnapshot,
)


## Step 1 - Start a connections-profile client

The `connections` profile groups the seven GB connection-intelligence tools. If the constructor raises, the usual cause is a missing `luminus-mcp` install on `PATH`.


In [ ]:
with Luminus(profile="connections", request_timeout=120.0) as lum:
    lum


## Step 2 - Run the site connection report

We use a coordinate near Berkswell GSP in the West Midlands. The `capacity_kind` value is narrative only; it does not affect scoring. Attaching a `project_name` gives the markdown summary a human-readable header.


In [ ]:
with Luminus(profile="connections", request_timeout=120.0) as lum:
    report: SiteConnectionReportSnapshot = lum.get_site_connection_report_snapshot(
        lat=52.39,
        lon=-1.64,
        capacity_kind="generation",
        project_name="Berkswell Alpha",
    )

print("queue:", report.traffic_lights.queue)
print("headroom:", report.traffic_lights.headroom)
print("land_constraints:", report.traffic_lights.land_constraints)


## Step 3 - Read the markdown summary

The `summary` field is pre-rendered markdown intended for analyst handoff. It includes the traffic-light banner, GSP context, DNO headroom site, and the land-constraint flags.


In [ ]:
print(report.summary)


## Step 4 - Inspect the canonical top TEC entries

The top entries come from the NESO TEC register (and NGED per-GSP queue rows where that GSP is covered), normalised into the canonical connection-entry schema. Each row keeps its source so you can audit provenance.


In [ ]:
for entry in report.structured.tec_queue.top_entries[:3]:
    print(
        f"[{entry.source}] site={entry.connection_site!r} "
        f"mw={entry.mw_capacity} stage={entry.lifecycle_stage}"
    )


## Step 5 - Inspect the canonical DNO headroom entry

When the nearest DNO site resolves to a supported operator (SSEN, NPG, UKPN, SPEN, ENWL), the report also carries a canonical representation of that row. Outside those operators the `canonical` field is `None`.


In [ ]:
dno = report.structured.dno_headroom
if dno.canonical is not None:
    c = dno.canonical
    print(
        f"operator={dno.operator} substation={c.connection_site!r} "
        f"mw={c.mw_capacity} stage={c.lifecycle_stage}"
    )
else:
    print(
        f"No canonical DNO entry. operator={dno.operator!r} substation={dno.substation!r}"
    )


## Step 6 - Review the confidence notes

Confidence notes surface upstream fetch failures and reiterate what the traffic lights do and do not say. Read these before quoting numbers in a memo.


In [ ]:
for note in report.confidence_notes:
    print("-", note)


## Step 7 - Pivot to the Gate 2 readiness check

The site report tells you what public data says about the coordinate. The Gate 2 readiness check asks whether a hypothetical project around that coordinate would pass the published Gate 2 entry criteria today. The two tools are complementary: use the report for the site, use the checklist for the project.


In [ ]:
with Luminus(profile="connections", request_timeout=120.0) as lum:
    readiness: Gate2ReadinessCheckSnapshot = lum.get_gate2_readiness_check_snapshot(
        project_name="Berkswell Alpha",
        technology="solar",
        capacity_mw=49.9,
        connection_voltage_kv=33,
        planning_status="submitted",
        land_rights_status="option",
        nominated_connection_point="BERKSWELL GSP",
        grid_reference="SP 243 778",
        target_energisation_year=2028,
    )


## Step 8 - Read the per-rule checklist

The summary counts pass / warn / fail / not_applicable rules. The per-rule results explain why a rule resolved the way it did and link to the published reference so you can audit the interpretation.


In [ ]:
s = readiness.summary
print(
    f"pass={s.pass_} warn={s.warn} fail={s.fail} "
    f"n/a={s.not_applicable} total={s.total}"
)
print()
for rule in readiness.results:
    print(f"[{rule.status.upper()}] {rule.rule_id}: {rule.reason}")
    print(f"    ref: {rule.reference_url}")


## Exercises

- Move the coordinate to a different GB region and see how the traffic lights and DNO operator change.
- Raise `capacity_mw` above 50 MW and re-run the Gate 2 check to see which threshold rules switch.
- Flip `planning_status` from `submitted` to `granted` and compare the resulting warn and fail counts.


## Pitfall

A `green` queue light only means the upstream signals positively reported zero entries. A missing or failed upstream returns `unknown`, which is not the same as clear. Always read `confidence_notes` alongside the traffic lights.


## Extension

If a site clears this pass, the next steps belong to professional due diligence: commission a formal grid study, engage the DNO and NESO, and progress planning and land control with qualified advisers. This notebook is a screening aid, not a substitute for that work.
